# 02 · Silver: limpieza y tipado

**Objetivo:** convertir las tablas crudas de `bronze` en tablas confiables en `silver`.

**Reglas de Silver que aplicamos:**
1. **Tipos correctos**: fechas como `timestamp`, importes como `decimal`, contadores como `int`.
2. **Sin duplicados**: nos quedamos con una fila por llave usando `row_number()`.
3. **Texto normalizado**: ciudades en formato título, estados en mayúscula, sin espacios sobrantes.
4. **Códigos postales con 5 dígitos**: al inferir tipos se perdieron los ceros a la izquierda (`01046` quedó `1046`); `lpad` los restaura.
5. **Calidad medible**: al final hay consultas que cuentan problemas.

Cada celda `%%sql` ejecuta **una** sentencia. Orden: pedidos -> ítems -> pagos -> reseñas -> clientes -> vendedores -> productos -> geolocalización -> verificaciones.

## 1. Pedidos (`silver.orders`)
- `to_timestamp()` convierte los textos de fecha a fecha-hora real (los vacíos pasan a `NULL`, normal en pedidos no entregados o cancelados).
- `row_number() OVER (PARTITION BY order_id ...)` numera las filas repetidas de un mismo pedido; con `WHERE rn = 1` conservamos solo una.
- Quitamos las columnas técnicas `_ingested_at` y `_source_file`.

In [ ]:
%%sql
CREATE OR REPLACE TABLE silver.orders AS
SELECT order_id,
customer_id,
order_status,
to_timestamp(order_purchase_timestamp) AS order_purchase_timestamp,
to_timestamp(order_approved_at) AS order_approved_at,
to_timestamp(order_delivered_carrier_date) AS order_delivered_carrier_date,
to_timestamp(order_delivered_customer_date) AS order_delivered_customer_date,
to_timestamp(order_estimated_delivery_date) AS order_estimated_delivery_date
FROM (
SELECT *,
row_number() OVER (PARTITION BY order_id ORDER BY _ingested_at DESC) AS rn
FROM bronze.orders
)
WHERE rn = 1

## 2. Líneas de pedido (`silver.order_items`)
- La llave es **`order_id` + `order_item_id`** (un pedido tiene varias líneas).
- `price` y `freight_value` pasan a `DECIMAL(12,2)`: para dinero nunca se usa `double`, por los errores de redondeo.

In [ ]:
%%sql
CREATE OR REPLACE TABLE silver.order_items AS
SELECT order_id,
CAST(order_item_id AS INT) AS order_item_id,
product_id,
seller_id,
to_timestamp(shipping_limit_date) AS shipping_limit_date,
CAST(price AS DECIMAL(12,2)) AS price,
CAST(freight_value AS DECIMAL(12,2)) AS freight_value
FROM (
SELECT *,
row_number() OVER (PARTITION BY order_id, order_item_id ORDER BY order_id) AS rn
FROM bronze.order_items
)
WHERE rn = 1

## 3. Pagos (`silver.order_payments`)
Un pedido puede pagarse con varios medios (`payment_sequential`), así que aquí **no** se deduplica por `order_id`. Solo se tipan las columnas.

In [ ]:
%%sql
CREATE OR REPLACE TABLE silver.order_payments AS
SELECT order_id,
CAST(payment_sequential AS INT) AS payment_sequential,
payment_type,
CAST(payment_installments AS INT) AS payment_installments,
CAST(payment_value AS DECIMAL(12,2)) AS payment_value
FROM bronze.order_payments

## 4. Reseñas (`silver.order_reviews`)
- Nos quedamos solo con puntuaciones válidas (1 a 5).
- En el dataset original algunos `review_id` aparecen repetidos: conservamos la respuesta más reciente (`ORDER BY review_answer_timestamp DESC`).

In [ ]:
%%sql
CREATE OR REPLACE TABLE silver.order_reviews AS
SELECT review_id,
order_id,
CAST(review_score AS INT) AS review_score,
review_comment_title,
review_comment_message,
to_timestamp(review_creation_date) AS review_creation_date,
to_timestamp(review_answer_timestamp) AS review_answer_timestamp
FROM (
SELECT *,
row_number() OVER (PARTITION BY review_id ORDER BY review_answer_timestamp DESC) AS rn
FROM bronze.order_reviews
WHERE CAST(review_score AS INT) BETWEEN 1 AND 5
)
WHERE rn = 1

## 5. Clientes (`silver.customers`)
- `customer_id` cambia en cada pedido; la persona real es `customer_unique_id` (se usará en Gold).
- `lpad(..., 5, '0')` devuelve el código postal a 5 dígitos.
- `initcap(trim(...))` pone la ciudad en formato título y `upper(trim(...))` el estado en mayúscula.

In [ ]:
%%sql
CREATE OR REPLACE TABLE silver.customers AS
SELECT customer_id,
customer_unique_id,
lpad(CAST(customer_zip_code_prefix AS STRING), 5, '0') AS customer_zip_code_prefix,
initcap(trim(customer_city)) AS customer_city,
upper(trim(customer_state)) AS customer_state
FROM (
SELECT *,
row_number() OVER (PARTITION BY customer_id ORDER BY customer_id) AS rn
FROM bronze.customers
)
WHERE rn = 1

## 6. Vendedores (`silver.sellers`)
Misma lógica que clientes: código postal a 5 dígitos, texto normalizado, una fila por `seller_id`.

In [ ]:
%%sql
CREATE OR REPLACE TABLE silver.sellers AS
SELECT seller_id,
lpad(CAST(seller_zip_code_prefix AS STRING), 5, '0') AS seller_zip_code_prefix,
initcap(trim(seller_city)) AS seller_city,
upper(trim(seller_state)) AS seller_state
FROM (
SELECT *,
row_number() OVER (PARTITION BY seller_id ORDER BY seller_id) AS rn
FROM bronze.sellers
)
WHERE rn = 1

## 7. Productos (`silver.products`)
- `LEFT JOIN` con `bronze.category_translation` para traer el nombre de categoría **en inglés**.
- `coalesce(...)` rellena con `'unknown'` los productos sin categoría, para que no desaparezcan en los reportes.
- Corregimos dos errores de ortografía del origen: `lenght` pasa a `length`.

In [ ]:
%%sql
CREATE OR REPLACE TABLE silver.products AS
SELECT p.product_id,
p.product_category_name,
coalesce(t.product_category_name_english, p.product_category_name, 'unknown') AS categoria,
CAST(p.product_name_lenght AS INT) AS product_name_length,
CAST(p.product_description_lenght AS INT) AS product_description_length,
CAST(p.product_photos_qty AS INT) AS product_photos_qty,
CAST(p.product_weight_g AS INT) AS product_weight_g
FROM (
SELECT *,
row_number() OVER (PARTITION BY product_id ORDER BY product_id) AS rn
FROM bronze.products
) p
LEFT JOIN bronze.category_translation t
ON p.product_category_name = t.product_category_name
WHERE p.rn = 1

## 8. Geolocalización (`silver.geolocation`)
La tabla original tiene ~1 millón de filas con **muchas coordenadas por cada código postal**. Para poder unirla con clientes y vendedores necesitamos **una fila por código postal**, así que agrupamos y promediamos latitud y longitud.

In [ ]:
%%sql
CREATE OR REPLACE TABLE silver.geolocation AS
SELECT lpad(CAST(geolocation_zip_code_prefix AS STRING), 5, '0') AS zip_code_prefix,
avg(CAST(geolocation_lat AS DOUBLE)) AS lat,
avg(CAST(geolocation_lng AS DOUBLE)) AS lng,
first(upper(trim(geolocation_state))) AS state
FROM bronze.geolocation
GROUP BY lpad(CAST(geolocation_zip_code_prefix AS STRING), 5, '0')

## 9. Verificación: conteos de filas
Compara con Bronze: `orders` y `customers` deben seguir en ~99.441, `order_items` en ~112.650, y `geolocation` baja de 1 millón a ~19.000 (una fila por código postal).

In [ ]:
%%sql
SELECT 'orders' AS tabla, COUNT(*) AS filas FROM silver.orders
UNION ALL SELECT 'order_items', COUNT(*) FROM silver.order_items
UNION ALL SELECT 'order_payments', COUNT(*) FROM silver.order_payments
UNION ALL SELECT 'order_reviews', COUNT(*) FROM silver.order_reviews
UNION ALL SELECT 'customers', COUNT(*) FROM silver.customers
UNION ALL SELECT 'sellers', COUNT(*) FROM silver.sellers
UNION ALL SELECT 'products', COUNT(*) FROM silver.products
UNION ALL SELECT 'geolocation', COUNT(*) FROM silver.geolocation

## 10. Verificación: reglas de calidad
Cada fila cuenta registros problemáticos. **Un valor distinto de 0 no siempre es un error**: hay que entenderlo y documentarlo, no esconderlo.
- *Pedidos sin ítems*: en Olist existen pedidos cancelados o no disponibles sin líneas; es un hallazgo real, no un fallo de carga.
- *Entrega anterior a la compra*: sería una fecha incoherente.

In [ ]:
%%sql
SELECT 'orders sin order_id' AS regla, COUNT(*) AS filas_con_problema
FROM silver.orders WHERE order_id IS NULL
UNION ALL
SELECT 'items con precio <= 0', COUNT(*)
FROM silver.order_items WHERE price <= 0
UNION ALL
SELECT 'items huerfanos (sin pedido)', COUNT(*)
FROM silver.order_items i LEFT ANTI JOIN silver.orders o ON i.order_id = o.order_id
UNION ALL
SELECT 'pedidos sin items', COUNT(*)
FROM silver.orders o LEFT ANTI JOIN silver.order_items i ON i.order_id = o.order_id
UNION ALL
SELECT 'entrega antes de la compra', COUNT(*)
FROM silver.orders WHERE order_delivered_customer_date < order_purchase_timestamp
UNION ALL
SELECT 'resenas sin pedido', COUNT(*)
FROM silver.order_reviews r LEFT ANTI JOIN silver.orders o ON r.order_id = o.order_id